In [1]:
import requests
import pandas as pd

In [2]:
state_fips = {
    'Alabama': '01', 'Alaska': '02', 'Arizona': '04', 'Arkansas': '05',
    'California': '06', 'Colorado': '08', 'Connecticut': '09', 'Delaware': '10',
    'District of Columbia': '11', 'Florida': '12', 'Georgia': '13', 'Hawaii': '15',
    'Idaho': '16', 'Illinois': '17', 'Indiana': '18', 'Iowa': '19',
    'Kansas': '20', 'Kentucky': '21', 'Louisiana': '22', 'Maine': '23',
    'Maryland': '24', 'Massachusetts': '25', 'Michigan': '26', 'Minnesota': '27',
    'Mississippi': '28', 'Missouri': '29', 'Montana': '30', 'Nebraska': '31',
    'Nevada': '32', 'New Hampshire': '33', 'New Jersey': '34', 'New Mexico': '35',
    'New York': '36', 'North Carolina': '37', 'North Dakota': '38', 'Ohio': '39',
    'Oklahoma': '40', 'Oregon': '41', 'Pennsylvania': '42', 'Rhode Island': '44',
    'South Carolina': '45', 'South Dakota': '46', 'Tennessee': '47', 'Texas': '48',
    'Utah': '49', 'Vermont': '50', 'Virginia': '51', 'Washington': '53',
    'West Virginia': '54', 'Wisconsin': '55', 'Wyoming': '56'
}

In [6]:
#not seasonally adjusted unemp series: LAUST{fips}0000000000003
series_ids = [f'LAUST{fips}0000000000003' for fips in state_fips.values()]

In [8]:
def fetch_bls_series(series_list, start_year, end_year):
    url = 'https://api.bls.gov/publicAPI/v2/timeseries/data/'
    payload = {
        'seriesid': series_list,
        'startyear': str(start_year),
        'endyear': str(end_year)
    }
    response = requests.post(url, json=payload)
    return response.json()

In [9]:
# Need Oct 2006 through Sep 2022
# BLS free API allows 20 years per call so split into two windows
# Also max 50 series per call so split states into two batches
batch1 = series_ids[:25]
batch2 = series_ids[25:]

print('Fetching 2006-2015...')
data_b1_early = fetch_bls_series(batch1, 2006, 2015)
data_b2_early = fetch_bls_series(batch2, 2006, 2015)

print('Fetching 2016-2022...')
data_b1_late = fetch_bls_series(batch1, 2016, 2022)
data_b2_late = fetch_bls_series(batch2, 2016, 2022)

print('Done fetching')

Fetching 2006-2015...
Fetching 2016-2022...
Done fetching


In [11]:
def parse_bls_monthly(data, state_fips):
    #Parse BLS API response to monthly state unemployment rates.
    fips_to_state = {v: k for k, v in state_fips.items()}
    rows = []

    for series in data.get('Results', {}).get('series', []):
        series_id = series['seriesID']
        fips = series_id[5:7]
        state = fips_to_state.get(fips)

        for item in series.get('data', []):
            if item['period'] == 'M13': #skip annual average M13
                continue 
            rows.append({
                'state': state,
                'year': int(item['year']),
                'month': int(item['period'].replace('M', '')),
                'unemp_rate': float(item['value'])
            })

    return pd.DataFrame(rows)
                                              

In [12]:
#parse all batches
df_monthly = pd.concat([
    parse_bls_monthly(data_b1_early, state_fips),
    parse_bls_monthly(data_b2_early, state_fips),
    parse_bls_monthly(data_b1_late, state_fips),
    parse_bls_monthly(data_b2_late, state_fips)], ignore_index=True)

df_monthly = df_monthly.sort_values(['state','year','month']).reset_index(drop=True)

print(f'Monthly rows: {len(df_monthly)}')
print(f'Expected: {51 * 12 * 17}')  # 51 states, 12 months, 17 years (2006-2022)
print()
print(df_monthly.head(15))

Monthly rows: 10200
Expected: 10404

      state  year  month  unemp_rate
0   Alabama  2006      1         4.6
1   Alabama  2006      2         4.5
2   Alabama  2006      3         4.1
3   Alabama  2006      4         3.7
4   Alabama  2006      5         3.6
5   Alabama  2006      6         4.4
6   Alabama  2006      7         4.4
7   Alabama  2006      8         4.2
8   Alabama  2006      9         3.8
9   Alabama  2006     10         3.6
10  Alabama  2006     11         3.7
11  Alabama  2006     12         3.6
12  Alabama  2007      1         4.3
13  Alabama  2007      2         4.2
14  Alabama  2007      3         3.8


In [17]:
#construct fiscal year average
def assign_fiscal_year(row):
    """
    Assign a fiscal year to each monthly observation.
    FY runs October(t-1) through September(t).
    So October, November, December belong to the NEXT fiscal year.
    
    Examples:
        Oct 2006 (month=10, year=2006) -> FY2007
        Sep 2007 (month=9,  year=2007) -> FY2007
        Oct 2007 (month=10, year=2007) -> FY2008
    """
    if row['month'] >= 10:
        return row['year'] + 1
    else:
        return row['year']


In [18]:
df_monthly['fiscal_year'] = df_monthly.apply(assign_fiscal_year, axis=1)

In [19]:
#keep only fiscal years 2007-2022
df_monthly = df_monthly[(df_monthly['fiscal_year'] >= 2007) & (df_monthly['fiscal_year'] <= 2022)]

In [20]:
#average 12 months within each state-fiscal year
df_unemp = (df_monthly.groupby(['state', 'fiscal_year'])['unemp_rate']
    .agg(unemp_rate_mean='mean', unemp_rate_min='min', unemp_rate_max='max', n_months='count').reset_index().rename(columns={'fiscal_year': 'Year'}))

In [21]:
print(f'Shape: {df_unemp.shape}')
print(f'Expected: {51 * 16}')
print()

# Check every state-year has exactly 12 months
incomplete = df_unemp[df_unemp['n_months'] != 12]
if len(incomplete) > 0:
    print(f'WARNING - incomplete months:')
    print(incomplete)
else:
    print('All state-years have exactly 12 months')

print()
print(df_unemp.head(10))

Shape: (800, 6)
Expected: 816

All state-years have exactly 12 months

     state  Year  unemp_rate_mean  unemp_rate_min  unemp_rate_max  n_months
0  Alabama  2007         3.941667             3.4             4.4        12
1  Alabama  2008         5.183333             3.9             6.5        12
2  Alabama  2009         8.833333             6.7            10.6        12
3  Alabama  2010        10.558333             9.9            11.6        12
4  Alabama  2011         9.883333             9.3            10.5        12
5  Alabama  2012         8.450000             7.7             9.0        12
6  Alabama  2013         7.475000             6.7             8.3        12
7  Alabama  2014         6.991667             6.3             7.5        12
8  Alabama  2015         6.150000             5.6             6.7        12
9  Alabama  2016         5.875000             5.3             6.3        12


In [22]:
df_unemp.to_csv('unemployment_fy2007_2022.csv', index=False)
print('Saved unemployment_fy2007_2022.csv')

Saved unemployment_fy2007_2022.csv


In [23]:
df_unemp[df_unemp['state'] == 'Wyoming']

,state,Year,unemp_rate_mean,unemp_rate_min,unemp_rate_max,n_months


In [24]:
'Wyoming' in df_unemp['state'].values

False

In [25]:
df_unemp[df_unemp['state'] == 'Wyoming'].shape[0]

0

In [28]:
# Fetch Wyoming separately
print('Fetching Wyoming...')
wyoming_data_early = fetch_bls_series(['LAUST560000000000003'], 2006, 2015)
wyoming_data_late = fetch_bls_series(['LAUST560000000000003'], 2016, 2022)

# Parse to monthly
df_wyoming_monthly_early = parse_bls_monthly(wyoming_data_early, state_fips)
df_wyoming_monthly_late = parse_bls_monthly(wyoming_data_late, state_fips)

df_wyoming_monthly = pd.concat([df_wyoming_monthly_early, df_wyoming_monthly_late], ignore_index=True)
print(f'Wyoming monthly rows: {len(df_wyoming_monthly)}')
print(df_wyoming_monthly.head())

Fetching Wyoming...
Wyoming monthly rows: 204
     state  year  month  unemp_rate
0  Wyoming  2015     12         4.8
1  Wyoming  2015     11         4.6
2  Wyoming  2015     10         4.1
3  Wyoming  2015      9         3.7
4  Wyoming  2015      8         3.8


In [33]:
# Add Wyoming to monthly data
df_monthly_complete = pd.concat([df_monthly, df_wyoming_monthly], ignore_index=True)

# Reassign fiscal year
df_monthly_complete['fiscal_year'] = df_monthly_complete.apply(assign_fiscal_year, axis=1)

# Filter to fiscal years 2007-2022
df_monthly_complete = df_monthly_complete[
    (df_monthly_complete['fiscal_year'] >= 2007) &
    (df_monthly_complete['fiscal_year'] <= 2022)
]

# Reconstruct fiscal year averages
df_unemp = (df_monthly_complete
    .groupby(['state', 'fiscal_year'])['unemp_rate']
    .agg(unemp_rate_mean='mean', unemp_rate_min='min', unemp_rate_max='max', n_months='count').reset_index().rename(columns={'fiscal_year': 'Year'}))

# Verify Wyoming is now there
print(f'Total states: {df_unemp.state.nunique()}')
print(f'Wyoming rows: {df_unemp[df_unemp.state == "Wyoming"].shape[0]}')
print()
print(df_unemp[df_unemp.state == 'Wyoming'])

Total states: 51
Wyoming rows: 16

       state  Year  unemp_rate_mean  unemp_rate_min  unemp_rate_max  n_months
800  Wyoming  2007         2.800000             2.1             3.6        12
801  Wyoming  2008         2.800000             2.2             3.5        12
802  Wyoming  2009         5.308333             2.8             6.5        12
803  Wyoming  2010         7.075000             6.0             8.5        12
804  Wyoming  2011         6.233333             5.5             7.3        12
805  Wyoming  2012         5.650000             4.6             6.5        12
806  Wyoming  2013         4.883333             4.0             6.1        12
807  Wyoming  2014         4.475000             3.8             5.3        12
808  Wyoming  2015         4.125000             3.7             4.8        12
809  Wyoming  2016         5.166667             4.1             5.9        12
810  Wyoming  2017         4.525000             3.6             5.7        12
811  Wyoming  2018         4.

In [34]:
df_unemp.shape

(816, 6)

In [35]:
df_unemp.to_csv('unemployment_fy2007_2022.csv', index=False)
print('Saved unemployment_fy2007_2022.csv')

Saved unemployment_fy2007_2022.csv
